## Pipeline Script 


Inputs: Group CEST and nmap data output from pyGluCEST as well as demographic data from _________. 
Outputs: Compiled dataframes with GluCEST and nmap data. Trimmed based on number of people with sufficient data.

    Trimmed subject-wise dfs: e.g., cestmat (outpath + 'trimmed_cestmat' + dataset + atlas + '.csv')
    Long form dfs: e.g., long_df (outpath + 'longform_grpdf' + dataset + '_' + atlas + '.csv')
         Also have version with standard nmap values
    Mean dfs: e.g., grouped_df (outpath + 'means_' + dataset + '_' + atlas + '.csv')


Ask Bryce for trimming help. How to threshold sample.

In [55]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


### Import Packages

In [1]:
import os
import glob
import numpy as np
import pandas as pd
#import network_fcon as fc
import scipy as sp
from scipy.stats import pearsonr
from scipy.stats import linregress
import seaborn as sns
import matplotlib.pyplot as plt
import re
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_cammoun2012

### Define paths and variables

In [2]:
# Set variables
dataset = 'longglucest_outputmeasures2'
atlas = 'atl-Cammoun2012_res-500'
nmaps = ["NMDA", "mGluR5", "GABA","CB1"]
maps = ["cest", "NMDA", "mGluR5", "GABA","CB1"]
normalize_cest = True

# Set paths
inpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/" + dataset
outpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas
outpath_box = '/Users/pecsok/Library/CloudStorage/Box-Box/GluCEST PhD/Manuscripts/Neuromaps/Results'
os.makedirs(os.path.join(outpath), exist_ok=True)

# Read in data
cestmat = pd.read_csv(inpath + "/all_subs_GluCEST_" + atlas + "_UNI.csv", sep=',')

# Set indices and correct column names
cestmat.set_index('Subject', inplace = True)
#dfs = [cestmat, NMDAmat, mGluR5mat, GABAmat]

# Load in standardized nmap data
# receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_scale1000_17.csv", sep=',')
receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_cammoun2012_scale500.csv", sep=',')


In [3]:
# Remove Subcortical ROIs
cam = fetch_cammoun2012()
info = pd.read_csv(cam['info'], sep=',')
info = info[info['scale']== 'scale500']
#labels = info['label'][info['structure']=='cortex'].values
#labels = info['label'][info['structure']=='cortex'].values + hemisphere
hemisphere = info['hemisphere'][info['structure']=='subcortex'].values
subcortex = info['label'][info['structure']=='subcortex'].values + hemisphere
print(subcortex.shape)

(15,)


In [4]:
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)
print(cestmat.shape)

(182, 725)


## Trim Data

In [5]:
# Remove parcels with < 20 voxels
for i, col in enumerate(cestmat.columns):
    if 'NZcount' in col:
        # Set mean col to nan
        mean_col = cestmat.columns[i - 1]
        sigma_col = cestmat.columns[i + 1]
        cestmat[mean_col] = np.where(cestmat[col] < 20, np.nan, cestmat[mean_col])
        cestmat[sigma_col] = np.where(cestmat[col] < 20, np.nan, cestmat[sigma_col])
        cestmat[col] = np.where(cestmat[col] < 20, np.nan, cestmat[col])       
print(cestmat.shape)

### NOTE about thresholding: These steps have been modified as a first-pass for thresholding. 
### Final thresholding step is now in R data curation after we filter to include only 1 scan per subject.
# Remove parcels with <50% of data.
columns = cestmat.columns[cestmat.notnull().sum() > len(cestmat)*.50]
# Trim all dfs based on column filter
cestmat= cestmat[columns]
print(cestmat.shape)


# ID subjects missing >65% of remaining GluCEST parcels
sparse_subjs = cestmat[cestmat.isna().sum(axis=1) > cestmat.shape[1] * 0.50].index
# Trim all dfs based on row filter
cestmat = cestmat.drop(index=sparse_subjs)
print(cestmat.shape)


# Remove subcortical ROIs
if atlas == 'atl-Cammoun2012_res-500':
    cestmat = cestmat.loc[:, ~cestmat.columns.str.contains('|'.join(subcortex), na=False)]
print(cestmat.shape)


# Save trimmed dfs
cestmat.to_csv(outpath + '/trimmed_cestmat' + dataset + atlas + '.csv', index=True)

(182, 725)
(182, 251)
(173, 251)
(173, 239)


## Normalize GluCEST

In [12]:
# Step 1: Select columns that contain 'NZMean'
if normalize_cest:
    nzmean_columns = [col for col in cestmat.columns if 'NZMean' in col]
    
    # Step 2: Calculate mean and std deviation for each subject (row-wise) across selected columns
    cestmat['Subject_Avg_NZMean'] = cestmat[nzmean_columns].mean(axis=1)
    cestmat['Subject_Std_NZMean'] = cestmat[nzmean_columns].std(axis=1)
    
    # Step 3: Calculate z-scores for all selected columns at once and store them in a new dataframe
    zscore_df = (cestmat[nzmean_columns].sub(cestmat['Subject_Avg_NZMean'], axis=0)
                 .div(cestmat['Subject_Std_NZMean'], axis=0))

    # Step 4: Concatenate the z-scores dataframe to the original cestmat dataframe
    cestmat = pd.concat([cestmat['group'], zscore_df], axis=1)
    print(cestmat)
    #cestmat.to_csv(outpath + '/grp_df_means_std_normalized_' + dataset + '_' + atlas + '.csv', index=False)


                group  frontalpole_1R NZMean  medialorbitofrontal_1R NZMean  \
Subject                                                                       
100522_12003    TD/NC              -1.523152                      -2.014883   
100522_12371    TD/NC              -1.912976                      -2.940456   
100522_12783    TD/NC              -2.268662                      -2.711424   
102041_12037  PRO/CHR              -0.619692                            NaN   
102041_12500  PRO/CHR               1.523190                       1.668420   
...               ...                    ...                            ...   
96902_11903     TD/NC              -0.791984                      -0.070269   
96902_12440     TD/NC              -2.293442                      -1.168977   
96902_12788     TD/NC              -0.845405                      -0.524194   
98370_12558     TD/NC               0.203037                      -0.787547   
98370_12952     TD/NC               0.394577        

# Use standard maps to make grp_df and long_df

In [10]:
# Read in receptor data
receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)
print(receptor_df)   

# Chop up receptor_df by map
NMDAmat = receptor_df[["Parcel","NMDA"]]
GABAmat = receptor_df[["Parcel","GABA"]]
mGluR5mat = receptor_df[["Parcel","mGluR5"]]
CB1mat = receptor_df[["Parcel","mGluR5"]]
#print(NMDAmat)

                        Parcel      NMDA    mGluR5      GABA       CB1
0      lateralorbitofrontal_9R -0.020868  0.164621 -0.150228  0.537246
1     lateralorbitofrontal_11R  0.366013 -0.085390  0.170483  0.814702
2      lateralorbitofrontal_5R  1.171849  1.015446  1.317157  0.908262
3      lateralorbitofrontal_6R  0.715088 -0.055909 -0.026400 -0.330707
4      lateralorbitofrontal_7R -0.125005  0.105069  0.344259  0.359209
...                        ...       ...       ...       ...       ...
1010                 pallidumL  1.184228 -2.200435 -3.248923  2.866536
1011            accumbensareaL  0.632115  0.061453 -0.237722  0.555997
1012              hippocampusL  0.855395 -1.009223 -1.039651  0.070223
1013                 amygdalaL  0.109529 -0.782412 -0.971487  1.043323
1014                brainstemL -0.798938 -4.706040 -5.108990 -3.239905

[1015 rows x 5 columns]


### Make classic grp_df

In [18]:
# Transpose receptor maps.
nmda = NMDAmat.T
gaba = GABAmat.T
mglur5 = mGluR5mat.T
cb1 = CB1mat.T

print(nmda)
# Keep only parcels contained in cestmat
cestmat_regions = [col.replace(' NZMean', '') for col in cestmat.columns if ' NZMean' in col]
nmda_filtered = nmda[[col for col in nmda.columns if col in cestmat_regions]]
gaba_filtered = gaba[[col for col in gaba.columns if col in cestmat_regions]]
mglur5_filtered = mglur5[[col for col in mglur5.columns if col in cestmat_regions]]
cb1_filtered = cb1[[col for col in cb1.columns if col in cestmat_regions]]

# Filtered columns
nmda_filtered.columns = [f"NMDA_{col}" for col in nmda_filtered.columns]
gaba_filtered.columns = [f"GABA_{col}" for col in gaba_filtered.columns]
mglur5_filtered.columns = [f"mGluR5_{col}" for col in mglur5_filtered.columns]
cb1_filtered.columns = [f"CB1_{col}" for col in cb1_filtered.columns]


# Repeat values for length of cestmat
nmda_repeated = pd.concat([nmda_filtered] * len(cestmat), ignore_index=True)
gaba_repeated = pd.concat([gaba_filtered] * len(cestmat), ignore_index=True)
mglur5_repeated = pd.concat([mglur5_filtered] * len(cestmat), ignore_index=True)
cb1_repeated = pd.concat([cb1_filtered] * len(cestmat), ignore_index=True)


#print(nmda_repeated)
# Concatenate
grp_df_std = pd.concat([cestmat, nmda_repeated, gaba_repeated, mglur5_repeated, cb1_repeated], axis=1)

# Save grp_df_std
#grp_df_std.to_csv(outpath + '/grp_df_std' + dataset + atlas + '.csv', index=True)
print(grp_df_std.head())

                           0                         1     \
Parcel  lateralorbitofrontal_9R  lateralorbitofrontal_11R   
NMDA                  -0.020868                  0.366013   

                           2                        3     \
Parcel  lateralorbitofrontal_5R  lateralorbitofrontal_6R   
NMDA                   1.171849                 0.715088   

                           4                         5     \
Parcel  lateralorbitofrontal_7R  lateralorbitofrontal_10R   
NMDA                  -0.125005                  0.039044   

                           6                         7     \
Parcel  lateralorbitofrontal_4R  lateralorbitofrontal_17R   
NMDA                   1.248462                   0.37993   

                           8                         9     ...       1005  \
Parcel  lateralorbitofrontal_8R  lateralorbitofrontal_15R  ...  insula_3L   
NMDA                  -0.529395                  0.130049  ...  -0.166495   

              1006             1007

### Make Longform df

In [19]:
# Get list of parcel names
parcels = cestmat.filter(like="NZMean").columns.tolist()
# Melt cestmat to get Glu data in long format
longdf_cest = cestmat.reset_index().melt(id_vars='Subject', value_vars=parcels, 
                                      var_name='Parcel', value_name='GluCEST')
# Rename Parcels
longdf_cest = longdf_cest.replace(' NZMean', '', regex=True)

# Merge with standard nmaps data
long_df_std = pd.merge(longdf_cest, NMDAmat, on=["Parcel"])
long_df_std = pd.merge(long_df_std, GABAmat, on = ["Parcel"])
long_df_std = pd.merge(long_df_std, mGluR5mat, on = ["Parcel"])
long_df_std = pd.merge(long_df_std, CB1mat, on = ["Parcel"])

print(long_df_std)
# Save the longform dataframe to a CSV
#long_df_std.to_csv(outpath + '/longform_grpdf_std_' + dataset + '_' + atlas + '.csv', index=False)
long_df_std.to_csv(outpath_box + '/longform_grpdf_std_' + dataset + '_' + atlas + 'NEWTHRESHOLD.csv', index=False)

            Subject          Parcel   GluCEST      NMDA      GABA  mGluR5_x  \
0      100522_12003  frontalpole_1R -1.523152 -2.137305 -1.099287 -0.306475   
1      100522_12371  frontalpole_1R -1.912976 -2.137305 -1.099287 -0.306475   
2      100522_12783  frontalpole_1R -2.268662 -2.137305 -1.099287 -0.306475   
3      102041_12037  frontalpole_1R -0.619692 -2.137305 -1.099287 -0.306475   
4      102041_12500  frontalpole_1R  1.523190 -2.137305 -1.099287 -0.306475   
...             ...             ...       ...       ...       ...       ...   
13662   96902_11903   precuneus_15R  1.087185  0.180437  1.082489  1.016334   
13663   96902_12440   precuneus_15R       NaN  0.180437  1.082489  1.016334   
13664   96902_12788   precuneus_15R       NaN  0.180437  1.082489  1.016334   
13665   98370_12558   precuneus_15R -0.032247  0.180437  1.082489  1.016334   
13666   98370_12952   precuneus_15R -0.371208  0.180437  1.082489  1.016334   

       mGluR5_y  
0     -0.306475  
1     -0.306475

In [15]:
means_df = (
    long_df_std.groupby(['Parcel'], as_index=False)
      .agg({
          'GluCEST': 'mean',
          'NMDA': 'mean',
          'GABA': 'mean',
          'mGluR5': 'mean',
          'CB1': 'mean'
      })
      .rename(columns={
          'GluCEST': 'CEST_avg',
          'NMDA': 'NMDA_avg',
          'GABA': 'GABA_avg',
          'mGluR5': 'mGluR5_avg',
          'CB1': 'CB1_avg'
      })
)

print(means_df)
# Display the collapsed dataframe
means_df.to_csv(outpath + '/means_df_std_' + dataset + '_' + atlas + 'NEWTHRESHOLD.csv', index=False)


KeyError: "Column(s) ['CB1'] do not exist"